1\. Environment Setup
---------------------

The authors utilized a Linux-based server with NVIDIA RTX GPUs. For this replication, we use **Detectron2**, a standard library for object detection research.

**Kaggle Note:** We install Detectron2 from source to ensure compatibility with Kaggle's pre-installed PyTorch version. We also ensure pyyaml is pinned to prevent dependency conflicts.

In [1]:
# Installation (Uncomment if needed)
# !pip install detectron2 -f https://dl.fbaipublicfiles.com/detectron2/wheels/cu111/torch1.9/index.html

!pip install effdet timm -q

import os
import copy
import torch
import numpy as np
import cv2
import glob
import matplotlib.pyplot as plt
from datetime import datetime
from PIL import Image



# Define Output Directory for Kaggle (Writable path)
OUTPUT_DIR = "/kaggle/working/output/naive_ddl"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Using Torch version: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 7.9 MB/s eta 0:00:00
Using Torch version: 2.8.0+cu126 | CUDA available: True


2\. Dataset Registration
------------------------

As per **Section 3.1**, the dataset consists of 5,900 images of paprika plants. We follow the paper's split:

*   **70% Training**
    
*   **10% Validation**
    
*   **20% Testing**
    

The DDL unit focuses on 6 abnormality categories found in the Paprika dataset.

In [2]:
import os
import cv2
import glob
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader

class PaprikaYoloDataset(Dataset):
    def __init__(self, img_dir, label_dir, image_size=(512, 512)):
        """
        Reads YOLO formatted annotations and outputs effdet compatible targets.
        """
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.image_size = image_size
        
        # Grab all image paths
        self.image_files = glob.glob(os.path.join(img_dir, "*.jpg")) + \
                           glob.glob(os.path.join(img_dir, "*.png"))

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, index):
        img_path = self.image_files[index]
        
        # 1. Load the Image
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        orig_h, orig_w = image.shape[:2]
        
        # Resize and normalize for EfficientDet
        image = cv2.resize(image, self.image_size)
        image = image.astype(np.float32) / 255.0
        
        # 2. Match the label file
        filename = os.path.basename(img_path)
        label_filename = os.path.splitext(filename)[0] + ".txt"
        label_path = os.path.join(self.label_dir, label_filename)
        
        bboxes = []
        classes = []
        
        # 3. Parse YOLO format: class_id, center_x, center_y, width, height
        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                lines = f.readlines()
                
            for line in lines:
                parts = line.strip().split()
                if len(parts) < 5: continue
                
                class_id = int(parts[0])
                cx, cy, w, h = map(float, parts[1:5])
                
                # YOLO coords are normalized [0, 1]. Convert to absolute pixel values
                # based on the *new* target image size (512x512)
                abs_cx = cx * self.image_size[0]
                abs_cy = cy * self.image_size[1]
                abs_w = w * self.image_size[0]
                abs_h = h * self.image_size[1]
                
                x_min = abs_cx - (abs_w / 2)
                y_min = abs_cy - (abs_h / 2)
                x_max = abs_cx + (abs_w / 2)
                y_max = abs_cy + (abs_h / 2)
                
                bboxes.append([x_min, y_min, x_max, y_max])
                # Note: effdet often requires 1-indexed classes (0 is background)
                # You may need to do `class_id + 1` depending on the effdet config version
                classes.append(class_id)
                
        # Handle background images
        if len(bboxes) == 0:
            bboxes = np.zeros((0, 4), dtype=np.float32)
            classes = np.zeros((0,), dtype=np.int64)
            
        # 4. Format target dict for effdet
        target = {
            'bboxes': torch.tensor(bboxes, dtype=torch.float32),
            'cls': torch.tensor(classes, dtype=torch.int64)
        }
        
        # Convert image to [C, H, W] tensor
        image = torch.tensor(image).permute(2, 0, 1)

        return image, target

# ==========================================
# Initialize the Training Dataset
# ==========================================
DATASET_ROOT = "/kaggle/input/datasets/tijesu26/paprika-dataset/data"
TRAIN_IMG_DIR = os.path.join(DATASET_ROOT, "train", "images")
TRAIN_LABEL_DIR = os.path.join(DATASET_ROOT, "train", "labels")

train_dataset = PaprikaYoloDataset(TRAIN_IMG_DIR, TRAIN_LABEL_DIR)

def collate_fn(batch):
    images, targets = tuple(zip(*batch))
    images = torch.stack(images)
    return images, targets

train_dataloader = DataLoader(
    train_dataset, 
    batch_size=4, 
    shuffle=True, 
    num_workers=2,
    collate_fn=collate_fn
)

print(f"YOLO Dataset loaded! Total training images: {len(train_dataset)}")

YOLO Dataset loaded! Total training images: 10342


In [3]:
# # Verification
# if "paprika_train" in DatasetCatalog.list():
#     dataset_dicts = DatasetCatalog.get("paprika_train")
#     if len(dataset_dicts) > 0:
#         d = dataset_dicts[20]
#         img = utils.read_image(d["file_name"], format="BGR")
#         visualizer = Visualizer(img[:, :, ::-1], metadata=MetadataCatalog.get("paprika_train"), scale=0.5)
                
#         # === REDUCE TEXT SIZE ===
#         # Manually override the calculated font size
#         # Try values between 10 (small) and 25 (large)
#         # visualizer._default_font_size = 15
        
#         out = visualizer.draw_dataset_dict(d)
#         plt.figure(figsize=(10, 10))
#         plt.imshow(out.get_image()[:, :, ::-1])
#         plt.title("Sample Data Verification")
#         plt.show()
#     else:
#         print("Dataset registered but no images found.")

## 3. Augmentation Strategy

We implement the specific augmentations mentioned in **Table 4** of the paper.
The authors noted that *Color Temperature* and *Noise* reduced performance, while *Geometric* transforms improved it.

**Implemented Transforms:**
1.  **Random Flip:** Probability 0.9 (Horizontal).
2.  **Scale/Resize:** Table 4 lists `Scale x=[0.8, 1.2]`. In Detectron2, we implement this via `ResizeShortestEdge` with a range of sizes to simulate multi-scale training around the 1024px baseline.


## 4. Model Configuration (Naïve DDL)
  - **Optimizer:** SGD
  - **Momentum:** 0.9
  - **Base LR:** 0.08 (with Cosine Decay)
  - **Weight Decay:** 0.0005
  - **Batch Size:** 16
  - **Max Iterations:** 100,000

<!-- end list -->

In [ ]:
import torch
from effdet import create_model_from_config, get_efficientdet_config
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR
from torch.cuda.amp import autocast, GradScaler

# ==========================================
# PHASE 3: Initialize the DIANA DDL Architecture
# ==========================================
def build_diana_ddl(num_classes=6, image_size=(512, 512)):
    print("Fetching EfficientDet-D3 config...")
    config = get_efficientdet_config('tf_efficientdet_d3')
    
    # 1. DO NOT change config.num_classes here!
    # Just update the image size
    config.image_size = image_size
    
    # 2. Pass num_classes directly into the creator function.
    # The library will now automatically handle chopping off the 810 layer
    # and replacing it with your 54 layer!
    model = create_model_from_config(
        config, 
        bench_task='train', 
        bench_labeler=True,
        num_classes=num_classes, # <--- The magic fix goes here
        pretrained=True, 
        pretrained_backbone=True
    )
    return model

diana_ddl = build_diana_ddl()
print("Figure 4 Architecture Loaded Successfully!")

# ==========================================
# PHASE 4: The Kaggle-Optimized Training Loop
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
diana_ddl.to(device)

# 1. Scaled Optimizer 
accumulation_steps = 8 # Update weights every 8 batches
effective_batch_size = 4 * accumulation_steps # Assuming DataLoader batch_size=4
adjusted_lr = 0.08 * (effective_batch_size / 64) # approx 0.04

optimizer = torch.optim.SGD(
    diana_ddl.parameters(), 
    lr=adjusted_lr, 
    momentum=0.9, 
    weight_decay=0.0005
)

# 2. Iteration Setup
total_iterations = 50000
warmup_iters = 1500
iters_per_epoch = len(train_dataloader) // accumulation_steps
num_epochs = (total_iterations // iters_per_epoch) + 1 

# 3. Schedulers (Same as Table 3)
warmup_scheduler = LinearLR(optimizer, start_factor=0.001, end_factor=1.0, total_iters=warmup_iters)
cosine_scheduler = CosineAnnealingLR(optimizer, T_max=(total_iterations - warmup_iters), eta_min=1e-6)
scheduler = SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_iters])

# 4. Initialize AMP Scaler for the Kaggle T4 GPU
scaler = GradScaler()

print(f"Starting Kaggle-Optimized training. Effective Batch Size: {effective_batch_size}")

global_step = 0 
diana_ddl.train()

for epoch in range(num_epochs):
    epoch_loss = 0.0
    optimizer.zero_grad() # Zero out gradients at the START of the epoch
    
    for batch_idx, (images, targets) in enumerate(train_dataloader):
        if global_step >= total_iterations: break
            
        images = images.to(device)
        
        # Pad targets
        max_boxes = max([len(t['bboxes']) for t in targets])
        if max_boxes == 0: max_boxes = 1 
        batch_bboxes = torch.zeros((len(targets), max_boxes, 4), device=device)
        batch_cls = torch.zeros((len(targets), max_boxes), dtype=torch.int64, device=device)
        
        for i, t in enumerate(targets):
            num_boxes = len(t['bboxes'])
            if num_boxes > 0:
                batch_bboxes[i, :num_boxes, :] = t['bboxes'].to(device)
                batch_cls[i, :num_boxes] = t['cls'].to(device)
                
        padded_targets = {'bbox': batch_bboxes, 'cls': batch_cls}
        
        # --- AMP Forward Pass ---
        with autocast():
            loss_dict = diana_ddl(images, padded_targets)
            loss = loss_dict['loss']
            # Divide loss by accumulation steps so gradients scale correctly
            loss = loss / accumulation_steps
        
        # --- AMP Backward Pass ---
        scaler.scale(loss).backward()
        
        # --- Accumulation Step ---
        if ((batch_idx + 1) % accumulation_steps == 0) or (batch_idx + 1 == len(train_dataloader)):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(diana_ddl.parameters(), max_norm=10.0)
            
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            
            global_step += 1
            
        # Re-multiply for accurate logging
        epoch_loss += (loss.item() * accumulation_steps)
        
        if batch_idx % 50 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] | Batch {batch_idx} | Loss: {loss.item() * accumulation_steps:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")
            
    print(f"=== Epoch {epoch+1} Complete | Average Loss: {epoch_loss / max(1, len(train_dataloader) // accumulation_steps):.4f} ===")
    if global_step >= total_iterations: break

torch.save(diana_ddl.state_dict(), "diana_ddl_kaggle_optimized.pth")

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Fetching EfficientDet-D3 config...
Downloading: "https://github.com/rwightman/efficientdet-pytorch/releases/download/v0.1/tf_efficientdet_d3_47-0b525f35.pth" to /root/.cache/torch/hub/checkpoints/tf_efficientdet_d3_47-0b525f35.pth
Figure 4 Architecture Loaded Successfully!
Starting Kaggle-Optimized training. Effective Batch Size: 32


/tmp/ipykernel_57/2219410729.py:63: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_57/2219410729.py:94: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [1/310] | Batch 0 | Loss: 2.3613 | LR: 0.000040
Epoch [1/310] | Batch 50 | Loss: 2.5547 | LR: 0.000120
Epoch [1/310] | Batch 100 | Loss: 2.3065 | LR: 0.000200
Epoch [1/310] | Batch 150 | Loss: 2.0675 | LR: 0.000280
Epoch [1/310] | Batch 200 | Loss: 2.5910 | LR: 0.000373
Epoch [1/310] | Batch 250 | Loss: 2.7514 | LR: 0.000453
Epoch [1/310] | Batch 300 | Loss: 2.2460 | LR: 0.000533
Epoch [1/310] | Batch 350 | Loss: 3.1232 | LR: 0.000613
Epoch [1/310] | Batch 400 | Loss: 2.9662 | LR: 0.000706
Epoch [1/310] | Batch 450 | Loss: 2.3601 | LR: 0.000786
Epoch [1/310] | Batch 500 | Loss: 2.3688 | LR: 0.000866
Epoch [1/310] | Batch 550 | Loss: 1.9148 | LR: 0.000946
Epoch [1/310] | Batch 600 | Loss: 2.2871 | LR: 0.001039
Epoch [1/310] | Batch 650 | Loss: 2.0460 | LR: 0.001119
Epoch [1/310] | Batch 700 | Loss: 2.5426 | LR: 0.001199
Epoch [1/310] | Batch 750 | Loss: 2.1963 | LR: 0.001279
Epoch [1/310] | Batch 800 | Loss: 2.3353 | LR: 0.001372
Epoch [1/310] | Batch 850 | Loss: 2.6299 | LR: 0.00

## Phase 5: The Test Data Loader (XML Format)

First, let's install torchmetrics and set up the dataloader that reads your XML test files. Run this in a new cell:

In [ ]:
!pip install torchmetrics -q

import os
import glob
import torch
import cv2
import xml.etree.ElementTree as ET
import numpy as np
from torch.utils.data import Dataset, DataLoader

# The EXACT class list from your notebook to ensure IDs match perfectly
CLASS_NAMES = [
    "blossom_end_rot", "graymold", "powdery_mildew", 
    "spider_mite", "spotting_disease", "snails_and_slugs"
]

class PaprikaXMLDataset(Dataset):
    def __init__(self, img_dir, xml_dir, image_size=(512, 512)):
        self.img_dir = img_dir
        self.xml_dir = xml_dir
        self.image_size = image_size
        self.image_files = glob.glob(os.path.join(img_dir, "*.jpg")) + glob.glob(os.path.join(img_dir, "*.png"))

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, index):
        img_path = self.image_files[index]
        
        # 1. Load and resize image
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        orig_h, orig_w = image.shape[:2]
        
        image = cv2.resize(image, self.image_size)
        image = image.astype(np.float32) / 255.0
        
        # 2. Parse XML
        filename = os.path.basename(img_path)
        xml_filename = os.path.splitext(filename)[0] + ".xml"
        xml_path = os.path.join(self.xml_dir, xml_filename)
        
        bboxes, classes = [], []
        
        if os.path.exists(xml_path):
            tree = ET.parse(xml_path)
            root = tree.getroot()
            
            for member in root.findall("object"):
                class_name = member.find("name").text
                if class_name not in CLASS_NAMES: continue
                    
                class_id = CLASS_NAMES.index(class_name)
                
                # Get raw XML coordinates
                bndbox = member.find("bndbox")
                xmin = float(bndbox.find("xmin").text)
                ymin = float(bndbox.find("ymin").text)
                xmax = float(bndbox.find("xmax").text)
                ymax = float(bndbox.find("ymax").text)
                
                # Scale coordinates to the new 512x512 size
                x_scale, y_scale = self.image_size[0] / orig_w, self.image_size[1] / orig_h
                bboxes.append([xmin * x_scale, ymin * y_scale, xmax * x_scale, ymax * y_scale])
                classes.append(class_id)
                
        # Handle empty images
        if not bboxes:
            bboxes = np.zeros((0, 4), dtype=np.float32)
            classes = np.zeros((0,), dtype=np.int64)
            
        target = {
            'boxes': torch.tensor(bboxes, dtype=torch.float32), # Note: torchmetrics uses 'boxes', not 'bboxes'
            'labels': torch.tensor(classes, dtype=torch.int64)  # Note: torchmetrics uses 'labels', not 'cls'
        }
        
        image = torch.tensor(image).permute(2, 0, 1)
        return image, target

# ==========================================
# Initialize the Test Dataset
# ==========================================
TEST_ROOT = "/kaggle/input/datasets/tijesu26/paprika-dataset/test_data"
TEST_IMG_DIR = os.path.join(TEST_ROOT, "images")
TEST_XML_DIR = os.path.join(TEST_ROOT, "xml")

# We use the same collate_fn as before
def collate_fn(batch):
    images, targets = tuple(zip(*batch))
    images = torch.stack(images)
    return images, targets

test_dataset = PaprikaXMLDataset(TEST_IMG_DIR, TEST_XML_DIR)
test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn)

print(f"Test Dataset loaded! Total test images: {len(test_dataset)}")

## Phase 6: The Evaluation Loop

During training, effdet wrapped our model in a DetBenchTrain module to calculate losses. For evaluation, we need to wrap it in a DetBenchPredict module, which takes the raw network outputs and performs Non-Maximum Suppression (NMS) to draw the final bounding boxes.

Here is the code to evaluate your trained model. You can run this directly after your training loop finishes:

In [ ]:
from effdet import DetBenchPredict
from torchmetrics.detection.mean_ap import MeanAveragePrecision

def evaluate_model(trained_train_bench, dataloader, device):
    print("Initializing Evaluation...")
    
    # 1. Extract the raw EfficientDet model from the training bench and wrap it for prediction
    eval_model = DetBenchPredict(trained_train_bench.model).to(device)
    eval_model.eval()
    
    # 2. Initialize TorchMetrics mAP calculator
    # This automatically computes COCO metrics including mAP@0.50 (what the paper uses)
    metric = MeanAveragePrecision(box_format='xyxy', class_metrics=True)
    
    with torch.no_grad():
        for batch_idx, (images, targets) in enumerate(dataloader):
            images = images.to(device)
            
            # effdet prediction expects image sizes/scales if we want to reverse resizing, 
            # but since our targets are also scaled to 512, we can compare directly!
            # The model outputs a tensor of shape [batch_size, num_detections, 6]
            # where the 6 values are [xmin, ymin, xmax, ymax, score, class_id]
            outputs = eval_model(images)
            
            # Format predictions for TorchMetrics
            preds = []
            for i in range(outputs.shape[0]):
                out = outputs[i]
                # Filter out padding/background detections (effdet pads empty detections with 0)
                valid_mask = out[:, 4] > 0.05 # Only keep boxes with > 5% confidence
                valid_out = out[valid_mask]
                
                preds.append({
                    "boxes": valid_out[:, 0:4].cpu(),
                    "scores": valid_out[:, 4].cpu(),
                    "labels": valid_out[:, 5].int().cpu()
                })
                
            # Format targets for TorchMetrics
            formatted_targets = []
            for t in targets:
                formatted_targets.append({
                    "boxes": t['boxes'].cpu(),
                    "labels": t['labels'].cpu()
                })
                
            # Add batch to the metric calculator
            metric.update(preds, formatted_targets)
            
            if batch_idx % 10 == 0:
                print(f"Evaluated Batch [{batch_idx}/{len(dataloader)}]")
                
    # Compute final metrics
    print("Computing final mAP scores...")
    results = metric.compute()
    
    # Print the specific metrics the paper cares about
    print("\n" + "="*40)
    print("FINAL EVALUATION METRICS (COCO Style)")
    print("="*40)
    print(f"mAP (IoU=0.50:0.95): {results['map'].item():.4f}")
    print(f"mAP (IoU=0.50):      {results['map_50'].item():.4f}  <-- Table 5 Metric")
    print(f"mAP (IoU=0.75):      {results['map_75'].item():.4f}")
    print("="*40)
    
    return results

# Assuming `diana_ddl` is your trained model and `device` is set
# Run the evaluation!
metrics = evaluate_model(diana_ddl, test_dataloader, device)